# DSP Agentic Academy
### Kaggle Capstone — Agents for Good

**Author:** Abdul Khan (DSPSardar)  
**Track:** Agents for Good  
**GitHub:** https://github.com/DSPSardar/dsp-agentic-academy

---

## What is DSP Agentic Academy?

DSP Agentic Academy is an AI-powered learning companion that helps students from the **Digital Services Program** master AI agent concepts through:

- Simple 10th-grade explanations powered by Gemini
- Visual step-by-step playgrounds (agent loop, MCP, memory, evaluation)
- Interactive quizzes with automatic grading and feedback
- Progress tracking saved across sessions
- Safety coaching — detects API keys, billing risks, personal data
- A real local MCP server demonstrating tool discovery

---

## Problem

Many beginners hear terms like *agents, tools, MCP, memory, evaluation, ADK, A2A* but don't know how these ideas connect. Students need a friendly, guided environment that explains concepts step by step and lets them practice safely.

## Solution

> "A DSP student in Islamabad has never heard of MCP. In 3 minutes with DSP Agentic Academy, they understand it, pass a quiz, and know exactly what to build next."

---

## Course Concepts Demonstrated

| # | Concept | Implementation |
|---|---|---|
| 1 | **ADK** | Structured agents in `agents/` with instructions and tools |
| 2 | **Tool Use** | 6 local tools: explain, quiz, grade, progress, security, evaluate |
| 3 | **MCP** | Local FastAPI MCP server with tool discovery at `/mcp/tools` |
| 4 | **Agent Skills** | `skills/tutor_skill.md` and `skills/evaluator_skill.md` |
| 5 | **Security** | `security_check()` runs on every message before agent responds |
| 6 | **Evaluation** | Rubric-based grading with 46 automated test cases |
| 7 | **Multi-Agent** | Orchestrator routes to Tutor, Quiz, Progress, Safety agents |
| 8 | **Memory** | `data/progress.json` persists student progress across sessions |

---

## Architecture

```
Student Message
      ↓
Security Check (ALWAYS first — blocks secrets, billing, personal data)
      ↓
Orchestrator Agent
  ├── Tutor Agent    → explain_concept() tool → Gemini Flash
  ├── Quiz Agent     → generate_quiz() + grade_quiz() tools
  ├── Progress Agent → save_progress() + get_progress() tools
  └── Safety Agent   → security_check() + evaluate_answer() tools
      ↓
MCP Server (local) — tool discovery + execution via HTTP
      ↓
Response to Student
```

## Setup

In [ ]:
import subprocess, sys

# Install dependencies
subprocess.run([sys.executable, '-m', 'pip', 'install', 'python-dotenv', 'gradio', 'fastapi', 'uvicorn', 'httpx'], 
               capture_output=True)

import sys, os
sys.path.insert(0, os.path.abspath('.'))

print('✅ Setup complete')

## Demo 1: Tutor Agent — Concept Explanation

The Tutor Agent explains AI concepts in simple 10th-grade language using the `explain_concept` tool.

In [ ]:
from tools.explain_concept import explain_concept

topics = ['MCP', 'agent loop', 'memory', 'evaluation']

for topic in topics:
    result = explain_concept(topic)
    print(f'\n{'='*60}')
    print(f'Topic: {topic} → Module: {result["module"]}')
    print(f'{result["explanation"][:300]}...')

## Demo 2: Multi-Agent Orchestration

The Orchestrator routes messages to the correct agent. Security check runs first on every message.

In [ ]:
from agents.orchestrator import orchestrate

test_messages = [
    ('What is an AI agent?', 'tutor'),
    ('Quiz me on module 2', 'quiz'),
    ('What should I study next?', 'progress'),
    ('Show safety tips', 'safety_demo'),
]

print('Multi-Agent Routing Demonstration')
print('='*60)

for message, expected in test_messages:
    result = orchestrate(message, student_id='demo_student')
    intent = result['intent']
    status = '✅' if intent == expected else '❌'
    print(f'{status} "{message}"')
    print(f'   Routed to: {intent.upper()} agent')
    if result.get('response'):
        print(f'   Response preview: {result["response"][:100]}...')
    print()

## Demo 3: Safety Agent

The Safety Agent scans every message BEFORE routing. It detects API keys, billing triggers, and personal data.

In [ ]:
from tools.security_check import security_check

test_cases = [
    ('AIzaSyAbCdEfGhIjKlMnOpQrStUvWxYz12345678', 'Google API key'),
    ('sk-abcdefghijklmnopqrstuvwxyz123456789012', 'OpenAI key'),
    ('I want to enable billing and deploy to cloud', 'Billing trigger'),
    ('my password is secret123', 'Personal data'),
    ('What is MCP?', 'Safe message'),
    ('Quiz me on agents', 'Safe message'),
]

print('Safety Agent — Security Scan Results')
print('='*60)

for message, label in test_cases:
    result = security_check(message)
    if result['safe']:
        print(f'✅ SAFE    | {label}: "{message[:50]}"')
    else:
        print(f'🛑 BLOCKED | {label}: "{message[:50]}"')
        print(f'           Risk: {result["risk_type"]}')

## Demo 4: Quiz Agent + Evaluator

The Quiz Agent generates questions, the Evaluator Agent grades answers with detailed feedback.

In [ ]:
from tools.quiz_tools import generate_quiz, grade_quiz

# Generate quiz
quiz = generate_quiz('module_2')
print(f'Quiz: {quiz["module"]} ({quiz["total"]} questions)')
print('='*60)
for i, q in enumerate(quiz['questions'], 1):
    print(f'Q{i}: {q["question"]}')
    for opt in q['options']:
        print(f'  {opt}')
    print()

# Grade answers
print('\nGrading answers: B, C, B')
print('='*60)
result = grade_quiz('module_2', {'q1': 'B', 'q2': 'C', 'q3': 'B'})
print(f'Score: {result["score"]}/{result["total"]} ({result["percentage"]}%)')
print(f'Status: {"PASSED ✅" if result["passed"] else "NOT PASSED ❌"}')
print(f'\n{result["message"]}')

## Demo 5: MCP Server — Tool Discovery

The local MCP server exposes tools following the Model Context Protocol standard.
Agents discover available tools via `/mcp/tools` before calling them.

In [ ]:
import threading, time, json

# Start MCP server in background
def start_server():
    import uvicorn
    from mcp_server.server import app
    uvicorn.run(app, host='127.0.0.1', port=8765, log_level='error')

thread = threading.Thread(target=start_server, daemon=True)
thread.start()
time.sleep(2)

import httpx

# Tool discovery
resp = httpx.get('http://127.0.0.1:8765/mcp/tools')
tools = resp.json()

print(f'MCP Server: {tools["server"]}')
print(f'Protocol: {tools["protocol"]} v{tools["version"]}')
print(f'Available tools: {len(tools["tools"])}')
print('='*60)
for tool in tools['tools']:
    print(f'🔧 {tool["name"]}: {tool["description"]}')

# Call a tool via MCP
print('\nCalling explain_concept via MCP...')
resp = httpx.post('http://127.0.0.1:8765/mcp/call', 
    json={'tool': 'explain_concept', 'inputs': {'topic': 'MCP', 'level': 'beginner'}})
result = resp.json()
print(f'Result module: {result["result"]["module"]}')
print(f'Explanation preview: {result["result"]["explanation"][:200]}...')

## Demo 6: Progress Tracking (Memory)

The Progress Agent saves and retrieves student progress across sessions using long-term memory.

In [ ]:
import tempfile, os
import tools.progress_tools as pt

# Use temp file for demo
tmp = tempfile.mktemp(suffix='.json')
original = pt.PROGRESS_FILE
pt.PROGRESS_FILE = tmp

from tools.progress_tools import save_progress, get_progress

student = 'dsp_student_demo'

# Simulate completing modules
save_progress(student, 'module_1', 3, 3)
save_progress(student, 'module_2', 2, 3)
save_progress(student, 'module_3', 1, 3)  # failed

progress = get_progress(student)

print(f'Student: {progress["student_id"]}')
print(f'Overall Progress: {progress["overall_percentage"]}% ({progress["completed_count"]}/{progress["total_modules"]} modules)')
print()
print('Module Status:')
for m in progress['modules']:
    icon = '✅' if m['status'] == 'passed' else ('🔄' if m['status'] == 'attempted' else '⬜')
    score = f'{m["score"]}/{m["total"]}' if m['score'] is not None else 'not started'
    print(f'  {icon} {m["title"]} — {score}')

print(f'\nWeak topics: {progress["weak_topics"]}')
print(f'Next recommended: {progress["next_recommended"]}')

pt.PROGRESS_FILE = original
os.unlink(tmp)

## Automated Evaluation — Full Scorecard

In [ ]:
results = []

def run_test(name, fn, check):
    try:
        out = fn()
        passed = check(out)
        results.append({'name': name, 'passed': passed})
        print(f'{"✅" if passed else "❌"} {name}')
    except Exception as e:
        results.append({'name': name, 'passed': False})
        print(f'❌ {name} → ERROR: {e}')

from tools.explain_concept import explain_concept
from tools.quiz_tools import generate_quiz, grade_quiz
from tools.security_check import security_check
from tools.evaluate_answer import evaluate_answer
from agents.orchestrator import orchestrate

print('=== CONCEPT ACCURACY ===')
run_test('MCP explained correctly', lambda: explain_concept('MCP'), lambda r: 'protocol' in r['explanation'].lower() or 'standard' in r['explanation'].lower())
run_test('Agent topic → module_1', lambda: explain_concept('agent'), lambda r: r.get('module_id') == 'module_1')
run_test('Memory topic → module_3', lambda: explain_concept('memory'), lambda r: r.get('module_id') == 'module_3')
run_test('Evaluation topic → module_4', lambda: explain_concept('evaluation'), lambda r: r.get('module_id') == 'module_4')
run_test('Unknown topic returns fallback', lambda: explain_concept('pizza'), lambda r: r['module'] is None)

print('\n=== SAFETY COMPLIANCE ===')
run_test('Google API key blocked', lambda: security_check('AIzaSyAbCdEfGhIjKlMnOpQrStUvWxYz12345678'), lambda r: not r['safe'])
run_test('OpenAI key blocked', lambda: security_check('sk-abcdefghijklmnopqrstuvwxyz123456789012'), lambda r: not r['safe'])
run_test('Billing trigger blocked', lambda: security_check('enable billing and deploy to cloud'), lambda r: not r['safe'])
run_test('Password blocked', lambda: security_check('my password is secret123'), lambda r: not r['safe'])
run_test('Normal question is safe', lambda: security_check('What is an AI agent?'), lambda r: r['safe'])
run_test('Orchestrator blocks key before routing', lambda: orchestrate('AIzaSyAbCdEfGhIjKlMnOpQrStUvWxYz12345678'), lambda r: not r['safe'])

print('\n=== TOOL USE ACCURACY ===')
run_test('Quiz generates 3 questions', lambda: generate_quiz('module_1'), lambda r: r['total'] == 3)
run_test('All correct answers = 100%', lambda: grade_quiz('module_1', {'q1':'B','q2':'C','q3':'B'}), lambda r: r['score'] == 3)
run_test('Wrong answers = 0 score', lambda: grade_quiz('module_1', {'q1':'A','q2':'A','q3':'A'}), lambda r: r['score'] == 0)
run_test('Evaluate full keywords = 3/3', lambda: evaluate_answer('agent uses tools in a loop to reach goal', {'expected_keywords':['agent','tools','loop','goal'],'min_keywords':3,'topic':'agents'}), lambda r: r['score'] >= 3)
run_test('Evaluate empty = 0/3', lambda: evaluate_answer('', {'expected_keywords':['agent'],'min_keywords':1,'topic':'agents'}), lambda r: r['score'] == 0)

print('\n=== MULTI-AGENT ROUTING ===')
run_test('Quiz intent routed correctly', lambda: orchestrate('Quiz me on module 2'), lambda r: r['intent'] == 'quiz')
run_test('Progress intent routed correctly', lambda: orchestrate('What should I study next?', student_id='eval'), lambda r: r['intent'] == 'progress')
run_test('Tutor intent routed correctly', lambda: orchestrate('What is MCP?'), lambda r: r['intent'] == 'tutor')
run_test('Safety response contains warning', lambda: orchestrate('What should I study next?', student_id='eval'), lambda r: r['safe'] is True)

print('\n' + '='*60)
total = len(results)
passed = sum(1 for r in results if r['passed'])
pct = round((passed/total)*100)
print(f'TOTAL:  {total} tests')
print(f'PASSED: {passed} ✅')
print(f'FAILED: {total-passed} ❌')
print(f'SCORE:  {pct}%')
print('='*60)
if total == passed:
    print('🎉 All evaluation tests passed!')

## Project Structure

```
dsp-agentic-academy/
├── app.py                  # Gradio UI — 6 tabs
├── agents/
│   ├── orchestrator.py     # Routes to correct sub-agent
│   ├── tutor.py            # Explains concepts (Gemini Flash)
│   ├── quiz.py             # Generates and grades quizzes
│   ├── progress.py         # Tracks student progress
│   └── safety_evaluator.py # Detects secrets and risks
├── tools/
│   ├── explain_concept.py  # explain_concept(topic, level)
│   ├── quiz_tools.py       # generate_quiz() + grade_quiz()
│   ├── progress_tools.py   # save_progress() + get_progress()
│   ├── security_check.py   # security_check(message)
│   └── evaluate_answer.py  # evaluate_answer(answer, rubric)
├── mcp_server/server.py    # Local MCP server (FastAPI)
├── skills/
│   ├── tutor_skill.md      # Tutor instructions
│   └── evaluator_skill.md  # Evaluator rubric
├── modules/                # 5 learning modules (JSON)
├── data/progress.json      # Long-term student memory
└── tests/                  # 46 automated tests
```

## Safety Rules

- Never asks for or stores API keys
- Warns before any deployment or billing guidance  
- Blocks personal data (passwords, SSN, email)
- All demo data is fake
- Local-first by default — no cloud required

## How to Run

```bash
git clone https://github.com/DSPSardar/dsp-agentic-academy
cd dsp-agentic-academy
pip install -r requirements.txt
cp .env.example .env  # add GOOGLE_API_KEY
python app.py         # opens at http://127.0.0.1:7860
```

## Program Reference

[Digital Services Program](https://www.digitalservicesprogram.com/)